### Set Explicit Data Directory

In [1]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / "data"

### Load Environment Variables

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

POSTGRES_USER = os.getenv("POSTGRES_USER")
POSTGRES_URL = os.getenv("POSTGRES_URL")

### Load PySpark including driver for postgresql

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("data-modeling-lab-bronze")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.4"
    )
    .getOrCreate()
)

In [4]:
print(spark.version)

3.5.4


### Write bronze_customers to parquet from data in postgresql

In [11]:
customers_df = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", "crm.customers")
    .option("user", POSTGRES_USER)
    .option("driver", "org.postgresql.Driver")
    .load()
)

customers_df.show()

+-----------+----------+---------+--------------------+---------+-----+-------------------+
|customer_id|first_name|last_name|               email|     city|state|         created_at|
+-----------+----------+---------+--------------------+---------+-----+-------------------+
|       1002|    Marcus|     Hill|marcus.hill@examp...|   Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|  Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|  Chicago|   IL|2026-03-08 16:20:00|
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|San Diego|   CA|2026-01-15 09:30:00|
+-----------+----------+---------+--------------------+---------+-----+-------------------+



In [12]:
customers_df.write.mode("overwrite").parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

In [13]:
bronze_customers = spark.read.parquet(
    str(DATA_DIR / "bronze" / "crm_customers")
)

bronze_customers.show()

+-----------+----------+---------+--------------------+---------+-----+-------------------+
|customer_id|first_name|last_name|               email|     city|state|         created_at|
+-----------+----------+---------+--------------------+---------+-----+-------------------+
|       1002|    Marcus|     Hill|marcus.hill@examp...|   Austin|   TX|2026-02-03 14:10:00|
|       1003|     Priya|     Shah|priya.shah@exampl...|  Seattle|   WA|2026-02-20 11:45:00|
|       1004|    Daniel|      Kim|daniel.kim@exampl...|  Chicago|   IL|2026-03-08 16:20:00|
|       1001|     Alice|   Nguyen|alice.nguyen@exam...|San Diego|   CA|2026-01-15 09:30:00|
+-----------+----------+---------+--------------------+---------+-----+-------------------+



### Write orders_bronze to parquet from postgresql

In [19]:
orders_df = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", "sales.orders")
    .option("user", POSTGRES_USER)
    .option("driver", "org.postgresql.Driver")
    .load()
)

orders_df.show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



In [20]:
orders_df.write.mode("overwrite").parquet(
    str(DATA_DIR / "bronze" / "sales_orders")
)

In [21]:
bronze_orders = spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_orders")
)

bronze_orders.show()

+--------+-----------+----------+------------+------------+
|order_id|customer_id|order_date|order_status|total_amount|
+--------+-----------+----------+------------+------------+
|    5001|       1001|2026-03-10|   Completed|      129.99|
|    5002|       1002|2026-03-11|   Completed|       84.50|
|    5003|       1001|2026-03-14|     Pending|       42.00|
|    5004|       1004|2026-03-18|   Completed|      215.75|
|    5005|       1003|2026-03-20|   Cancelled|       59.99|
+--------+-----------+----------+------------+------------+



### Write bronze_order_items to parquet from postgresql

In [22]:
order_items_df = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", "sales.order_items")
    .option("user", POSTGRES_USER)
    .option("driver", "org.postgresql.Driver")
    .load()
)

order_items_df.show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    5001|      2001|       1|     34.99|
|    5001|      2002|       1|     89.99|
|    5002|      2003|       2|     44.99|
|    5003|      2004|       1|     54.99|
|    5004|      2002|       2|     89.99|
|    5005|      2001|       1|     34.99|
+--------+----------+--------+----------+



In [23]:
order_items_df.write.mode("overwrite").parquet(
    str(DATA_DIR / "bronze" / "sales_order_items")
)

In [24]:
spark.read.parquet(
    str(DATA_DIR / "bronze" / "sales_order_items")
).show()

+--------+----------+--------+----------+
|order_id|product_id|quantity|unit_price|
+--------+----------+--------+----------+
|    5001|      2001|       1|     34.99|
|    5001|      2002|       1|     89.99|
|    5002|      2003|       2|     44.99|
|    5003|      2004|       1|     54.99|
|    5004|      2002|       2|     89.99|
|    5005|      2001|       1|     34.99|
+--------+----------+--------+----------+



### Write erp_products to parquet from postgresql

In [25]:
products_df = (
    spark.read
    .format("jdbc")
    .option("url", POSTGRES_URL)
    .option("dbtable", "erp.products")
    .option("user", POSTGRES_USER)
    .option("driver", "org.postgresql.Driver")
    .load()
)

products_df.show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



In [26]:
products_df.write.mode("overwrite").parquet(
    str(DATA_DIR / "bronze" / "erp_products")
)

In [27]:
spark.read.parquet(
    str(DATA_DIR / "bronze" / "erp_products")
).show()

+----------+-------------------+-----------+---------+----------+
|product_id|       product_name|   category|unit_cost|unit_price|
+----------+-------------------+-----------+---------+----------+
|      2001|     Wireless Mouse|Electronics|    18.00|     34.99|
|      2002|Mechanical Keyboard|Electronics|    52.00|     89.99|
|      2003|       Laptop Stand|     Office|    21.50|     44.99|
|      2004|          USB-C Hub|Electronics|    28.00|     54.99|
+----------+-------------------+-----------+---------+----------+



### Parquet Validation

In [28]:
for path in [
    "crm_customers",
    "sales_orders",
    "sales_order_items",
    "erp_products",
]:
    df = spark.read.parquet(str(DATA_DIR / "bronze" / path))
    print(path, df.count())

crm_customers 4
sales_orders 5
sales_order_items 6
erp_products 4
